# Summary

Search the Bedrock Knowledge base

In [1]:
import os, sys
import pandas as pd
import re
import json
import time

# AWS Python
import boto3

utils_path = "/Users/stephengodfrey/Documents/Workbench/Numantic/utilities/.."
sys.path.insert(0, utils_path)
from utilities.osa_tools.authentication import ApiAuthentication

api_configs = ApiAuthentication(client="Numantic")


## Construct a AWS Bedrock Retrieval and Generate class object

In [2]:
class BedrockKBRetriever:
    """
    Class to demonstrate AWS Bedrock Knowledge Base retrieve and retrieve and generate results.
    """

    def __init__(self,
                 aws_profile: str,
                 aws_region: str,
                 kb_id: str):

        self.aws_profile = aws_profile
        self.aws_region = aws_region
        self.kb_id = kb_id

        # Retrieval configuration
        self.num_srch_res = 5
        self.s3_bucket = "rag-search-tests"
        self.s3_doc_loc = "s3://{}/".format(self.s3_bucket)

        # Retrieval and Generation configuration
        self.aws_account_id = os.environ["AWS_ACCOUNT_ID"]
        # self.inference_model = "amazon.nova-lite-v1:0" #?
        # self.inference_model = "anthropic.claude-sonnet-4-6" # Requires Anthropic request form
        # self.inference_model = "google.gemma-3-4b-it" # Not available
        # self.inference_model = "meta.llama4-scout-17b-instruct-v1:0"
        self.inference_model = "anthropic.claude-sonnet-4-5-20250929-v1:0"
        # self.inference_model = "amazon.nova-pro-v1:0"
        # self.inference_model = "google.gemma-3-12b-it"
        self.model_arn = "arn:aws:bedrock:{}:{}:inference-profile/us.{}".format(self.aws_region,
                                                                                self.aws_account_id,
                                                                                self.inference_model)
        self.search_sleep_time = 1.5 # pause time between queries
        self.ai_sleep_time = 1.5 # pause time between queries

        # Outputs
        self.df_ret_res = pd.DataFrame()
        self.df_query_finds = pd.DataFrame()
        self.df_rag_res = pd.DataFrame()
        self.bedrock_models = pd.DataFrame() # AWS foundation models - requires get_bedrock_foundational_models method
        self.rag_responses = []

        # Establish an AWS client
        self.set_aws_client()

    def set_aws_client(self):
        """
        Create a AWS client
        :return:
        """

        # Initialize Bedrock Agent Runtime client for querying
        session = boto3.Session(profile_name=self.aws_profile)

        # Bedrock client
        self.br_rt = session.client('bedrock-agent-runtime',
                                    region_name=self.aws_region)

        # S3 client
        self.s3 = session.client('s3',
                                 region_name=self.aws_region)

    def get_bedrock_foundational_models(self):
        """
        Get a list of foundation models available through Bedrock
        :return:
        """

        # Initialize a bedrock client
        session = boto3.Session(profile_name=self.aws_profile)
        self.br = session.client('bedrock', region_name=self.aws_region)

        response = self.br.list_foundation_models()
        self.response_models = response

        models_rows = []
        for model in response['modelSummaries']:
            models_rows.append(dict(model_name=model['modelName'],
                                    model_id=model['modelId'],
                                    model_provider=model['providerName'],
                                    input_modalities=model['inputModalities'],
                                    output_modalities=model['outputModalities'],
                                    infer_types=model['inferenceTypesSupported'],
                                    model_lifecycle=model['modelLifecycle']
                                    )
                               )

        self.bedrock_models = pd.DataFrame(data=models_rows)
        self.bedrock_models = self.bedrock_models.sort_values(by="model_provider")
        self.bedrock_models = self.bedrock_models.reset_index(drop=True)

    def retrieve_query_results(self,
                               queries: list,
                               source_types: list = []):
        """
        Return AWS Bedrock search results for a list of queries. Filtering before searching can be
        applied by passing source type values in the source_types input parameter.
        :return:
        """

        # Set retrieval configurations
        if len(source_types) == 0:
            ret_config = {'vectorSearchConfiguration': {
                                                        'numberOfResults': self.num_srch_res
                                                        }
                         }

        if len(source_types) > 0:
            ret_config = {'vectorSearchConfiguration': {
                                                        'numberOfResults': self.num_srch_res,
                                                        'filter': {
                                                            'in': {
                                                                'key': 'source_type',
                                                                'value': source_types
                                                            }
                                                        }
                                                        }
                         }

        search_results = []
        for query in queries:

            # Set retrieval configurations
            ret_query_param = {'text': query
                               }

            response = self.br_rt.retrieve(knowledgeBaseId=self.kb_id,
                                           retrievalQuery=ret_query_param,
                                           retrievalConfiguration=ret_config
                                           )

            for result in response["retrievalResults"]:

                # Get a list of passages
                regex_pattern = r"\[(\d+)\]"

                passages = re.findall(regex_pattern, result['content']['text'], flags=re.DOTALL)

                # Clean up whitespace from the results
                passages = [p.strip() for p in passages]

                # Get metadata
                s3_loc=result['location']['s3Location']['uri']
                doc_name = s3_loc.replace(self.s3_doc_loc, "")
                key = "{}.metadata.json".format(doc_name)

                # Read the file from S3
                response_docread = self.s3.get_object(Bucket=self.s3_bucket,
                                                      Key=key)
                content = response_docread['Body'].read().decode('utf-8')

                # Parse JSON into dictionary
                metadata_dict = json.loads(content)
                source_type = metadata_dict["metadataAttributes"]["source_type"]

                res_dict = dict(query=query,
                                search_res=result['content']['text'],
                                passages=passages,
                                score=result['score'],
                                s3_loc=result['location']['s3Location']['uri'],
                                source_type=source_type,
                                doc_metadata=metadata_dict
                                )

                search_results.append(res_dict)

                time.sleep(self.search_sleep_time)

        # Put the detailed retrieval results into a dataframe
        self.df_ret_res = pd.DataFrame(search_results)

        # Aggregate results by query
        self.aggregate_retrieval_results()

    def aggregate_retrieval_results(self):
        """
        Aggregate retrieval results by query
        :param queries:
        :return:
        """

        q_rows = []
        for query in self.df_ret_res["query"].unique():

            mask = self.df_ret_res["query"] == query

            doc_value_cnts = self.df_ret_res[mask]["s3_loc"].value_counts()

            # Get passages
            pass_dict = {}
            for idx in self.df_ret_res[mask].index:
                doc_filename = self.df_ret_res.loc[idx, "s3_loc"].replace("{}documents/".format(self.s3_doc_loc),
                                                                          "")

                if doc_filename in pass_dict:
                    pass_dict[doc_filename] = pass_dict[doc_filename] + self.df_ret_res.loc[idx, "passages"]
                else:
                    pass_dict[doc_filename] = self.df_ret_res.loc[idx, "passages"]

            for df in pass_dict:
                pass_dict[df] = list(set(pass_dict[df]))
                pass_dict[df].sort()

            doc_dict = {}
            for s3_loc_idx in doc_value_cnts.index:
                ikey = s3_loc_idx.replace("{}documents/".format(self.s3_doc_loc), "")
                doc_dict[ikey] = int(doc_value_cnts[s3_loc_idx])

            doc_count = len(doc_dict)

            max_score = self.df_ret_res[mask]["score"].max()

            source_types = self.df_ret_res[mask]["source_type"].unique().tolist()

            q_rows.append(dict(query=query,
                               doc_count=doc_count,
                               max_score=max_score,
                               doc_scores=doc_dict,
                               passages=pass_dict,
                               source_types=source_types
                               )
                          )

        self.df_query_finds = pd.DataFrame(q_rows)

    def retrieve_and_generate_results(self,
                                      queries: list,
                                      source_types: list = []):
        """
        Retrieve AWS Bedrock search results and for a list of queries and generate AI responses summarizing the findings. Filtering before searching can be
        applied by passing source type values in the source_types input parameter.
        :return:
        """

        # Set retrieval configurations
        if len(source_types) == 0:
            ret_gen_config = {'type': 'KNOWLEDGE_BASE',
                              'knowledgeBaseConfiguration': {'knowledgeBaseId': self.kb_id,
                                                             'modelArn': self.model_arn
                                                             }
                              }

        if len(source_types) > 0:
              ret_gen_config = {'type': 'KNOWLEDGE_BASE',
                              'knowledgeBaseConfiguration': {'knowledgeBaseId': self.kb_id,
                                                             'modelArn': self.model_arn,
                                                             'retrievalConfiguration': {
                                                                'vectorSearchConfiguration': {
                                                                    'numberOfResults': self.num_srch_res,
                                                                    'filter': {
                                                                        'in': {
                                                                            'key': 'source_type',
                                                                            'value':  source_types
                                                                        }
                                                                    }
                                                                }
                                                             }
                                                             }
                                }

        search_results = []
        for query in queries:

            rag_response = self.br_rt.retrieve_and_generate(input={'text': query},
                                                            retrieveAndGenerateConfiguration=ret_gen_config
                                                            )

            self.rag_responses.append(rag_response)
            rag_answer = rag_response['output']['text']

            citations_list = []
            for citation in rag_response['citations']:
                for reference in citation['retrievedReferences']:
                    if 'location' in reference:
                        if 's3Location' in reference['location']:
                            citations_list.append(reference['location']['s3Location']['uri'])

            search_results.append(dict(query=query,
                                       rag_answer=rag_answer,
                                       rag_citations=citations_list)
                                    )

            time.sleep(self.ai_sleep_time)

        self.df_rag_res = pd.DataFrame(search_results)


## Read test data

In [3]:
input_data_path = "../data/rag_eval_dataset"
docs_filename = "documents.csv"
multi_pas_qs = "multi_passage_answer_questions.csv"
no_answer_qs = "no_answer_questions.csv"
single_pas_answer_qs = "single_passage_answer_questions.csv"

df_docs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, docs_filename))
df_mpqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, multi_pas_qs))
df_noaqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, no_answer_qs))
df_spqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, single_pas_answer_qs))


## Build question-answer and question-source-doc maps for single passage answers

In [4]:
# Question to source document index
sp_q_doc_index_map = {k:v for k, v in zip(df_spqs["question"],
                                          df_spqs["document_index"])}
sp_q_doc_index_map = {k: "doc_{}.txt".format(v) for k, v in zip(sp_q_doc_index_map.keys(),
                                                                sp_q_doc_index_map.values())}

# Question to source document index
sp_q_answer_map = {k:v for k, v in zip(df_spqs["question"],
                                       df_spqs["answer"])}
# sp_q_doc_index_map
# sp_q_answer_map


In [5]:
# df_docs


## Get available foundational models

In [6]:
# Set up some parameters
aws_profile = "ns-admin"
aws_region = 'us-east-2'
kb_id = "365JAS6WGF"


In [7]:
# Set up retriever object
srch_analyzer = BedrockKBRetriever(aws_profile=aws_profile,
                                   aws_region=aws_region,
                                   kb_id=kb_id)

# Set queries
srch_analyzer.get_bedrock_foundational_models()


In [8]:
# srch_analyzer.bedrock_models.loc[[20]].T
# srch_analyzer.bedrock_models.loc[[31]].T
srch_analyzer.bedrock_models.head()

# srch_analyzer.response_models



,model_name,model_id,model_provider,input_modalities,output_modalities,infer_types,model_lifecycle
0,Nova Premier,amazon.nova-premier-v1:0:mm,Amazon,"[TEXT, IMAGE, VIDEO]",[TEXT],[],"{'status': 'LEGACY', 'endOfLifeTime': 2026-09-..."
1,Nova Micro,amazon.nova-micro-v1:0,Amazon,[TEXT],[TEXT],[INFERENCE_PROFILE],"{'status': 'ACTIVE', 'startOfLifeTime': 2024-1..."
2,Titan Text Embeddings V2,amazon.titan-embed-text-v2:0,Amazon,[TEXT],[EMBEDDING],[ON_DEMAND],"{'status': 'ACTIVE', 'startOfLifeTime': 2024-0..."
3,Nova Premier,amazon.nova-premier-v1:0:8k,Amazon,"[TEXT, IMAGE, VIDEO]",[TEXT],[],"{'status': 'LEGACY', 'endOfLifeTime': 2026-09-..."
4,Nova Premier,amazon.nova-premier-v1:0:20k,Amazon,"[TEXT, IMAGE, VIDEO]",[TEXT],[],"{'status': 'LEGACY', 'endOfLifeTime': 2026-09-..."


## Retrieve Search Results

In [9]:
# Set up retriever object
srch_analyzer = BedrockKBRetriever(aws_profile=aws_profile,
                                   aws_region=aws_region,
                                   kb_id=kb_id)

# Set queries
queries = df_spqs["question"].tolist()

# Search documents for text related to queries without filtering
srch_analyzer.retrieve_query_results(queries=queries)

# Search documents for text related to queries with filtering
# srch_analyzer.retrieve_query_results(queries=queries,
#                                      source_types=["gaming"])

In [10]:
# Add source document and answer columns
df_query_finds = srch_analyzer.df_query_finds.copy(deep=True)
df_query_finds["source_document_index"] = df_query_finds["query"].map(sp_q_doc_index_map)
df_query_finds["answer"] = df_query_finds["query"].map(sp_q_answer_map)


srch_analyzer.df_ret_res
srch_analyzer.df_query_finds
# srch_analyzer.response


# sp_q_doc_index_map
# sp_q_answer_map

qf_cols = ['query', 'doc_count', 'max_score', 'doc_scores',
           'source_document_index', 'answer', 'source_types', 'passages']
df_query_finds = df_query_finds[qf_cols]
df_query_finds


,query,doc_count,max_score,doc_scores,source_document_index,answer,source_types,passages
0,What do keybullet kin drop?,1,0.708868,{'doc_0.txt': 5},doc_0.txt,Keybullet kin drop a key upon death.,[gaming],"{'doc_0.txt': ['1', '2', '23', '24', '25', '26..."
1,What kind of gun does the bandana bullet kin use?,1,0.579380,{'doc_0.txt': 5},doc_0.txt,The bandana bullet kin wields a machine pistol.,[gaming],"{'doc_0.txt': ['1', '10', '11', '12', '13', '1..."
2,What do the giants look like?,3,0.508541,"{'doc_1.txt': 2, 'doc_15.txt': 2, 'doc_0.txt': 1}",doc_1.txt,"One giant is burly, grey-skinned, and 20 feet ...","[gaming, entertainment]","{'doc_1.txt': ['10', '11', '12', '13', '14', '..."
3,What happens on day 2?,3,0.376101,"{'doc_18.txt': 2, 'doc_16.txt': 2, 'doc_9.txt'...",doc_1.txt,"After a few miles of winding tunnel, you emerg...","[entertainment, gaming]","{'doc_18.txt': ['77', '78'], 'doc_16.txt': [],..."
4,What were the requirements for the project?,4,0.388976,"{'doc_18.txt': 2, 'doc_5.txt': 1, 'doc_2.txt':...",doc_2.txt,The tool had the following requirements:\n- Ch...,"[data_science, government, entertainment]","{'doc_5.txt': ['38', '39'], 'doc_2.txt': ['13'..."
5,What data did was used to test the prototype?,1,0.439442,{'doc_2.txt': 5},doc_2.txt,Grace Hopper's Wikipedia page and Alan Turing'...,[data_science],"{'doc_2.txt': ['17', '18', '19', '20', '21', '..."
6,How do the data storage options compare?,4,0.425207,"{'doc_2.txt': 2, 'doc_3.txt': 1, 'doc_11.txt':...",doc_3.txt,For fast start: use SQLite3 and ChromaDB (File...,"[data_science, gaming]","{'doc_3.txt': ['36'], 'doc_11.txt': ['44', '45..."
7,When was UTF-8 support added for European lang...,2,0.384348,"{'doc_16.txt': 4, 'doc_3.txt': 1}",doc_3.txt,UTF-8 support was added for European languages...,"[data_science, gaming]","{'doc_3.txt': ['51', '52', '53'], 'doc_16.txt'..."
8,How do I make a button?,1,0.417596,{'doc_4.txt': 5},doc_4.txt,import marimo as mo\n\nbutton = mo.ui.run_butt...,[recipes],"{'doc_4.txt': ['100', '101', '50', '51', '52',..."
9,When might I use caching?,4,0.412053,"{'doc_19.txt': 2, 'doc_4.txt': 1, 'doc_11.txt'...",doc_4.txt,"You might use caching when, for example, your ...","[recipes, data_science]","{'doc_4.txt': ['100', '101', '102', '103', '10..."


## Retrieve and Generate AI Responses using RAG

In [11]:
# Set up retriever object
rag_analyzer = BedrockKBRetriever(aws_profile=aws_profile,
                                   aws_region=aws_region,
                                   kb_id=kb_id)

# Set queries
queries = df_spqs["question"].tolist()

# Retrieve RAG responses for text related to queries without filtering
rag_analyzer.retrieve_and_generate_results(queries=queries)

# Search RAG responses for text related to queries with filtering
# rag_analyzer.retrieve_and_generate_results(queries=queries,
#                                            source_types=["gaming"])


In [12]:

rag_analyzer.df_rag_res.loc[3,"rag_answer"]

rag_analyzer.df_rag_res



,query,rag_answer,rag_citations
0,What do keybullet kin drop?,Keybullet Kin drop a key upon death. Jammed Ke...,[s3://rag-search-tests/documents/doc_0.txt]
1,What kind of gun does the bandana bullet kin use?,The Bandana Bullet Kin uses a Machine Pistol.,[s3://rag-search-tests/documents/doc_0.txt]
2,What do the giants look like?,There are two giants described. The first gian...,"[s3://rag-search-tests/documents/doc_1.txt, s3..."
3,What happens on day 2?,"Sorry, I am unable to assist you with this req...",[]
4,What were the requirements for the project?,The required features for the project were:\n-...,[s3://rag-search-tests/documents/doc_2.txt]
5,What data did was used to test the prototype?,The prototype was not tested with any specific...,"[s3://rag-search-tests/documents/doc_2.txt, s3..."
6,How do the data storage options compare?,The data storage options are organized into di...,[s3://rag-search-tests/documents/doc_3.txt]
7,When was UTF-8 support added for European lang...,UTF-8 encoding for European languages was adde...,[s3://rag-search-tests/documents/doc_3.txt]
8,How do I make a button?,"To make a button in marimo, use `mo.ui.button(...","[s3://rag-search-tests/documents/doc_4.txt, s3..."
9,When might I use caching?,You might use caching when you have expensive ...,[s3://rag-search-tests/documents/doc_4.txt]


In [50]:
rag_analyzer.rag_responses[3]

{'ResponseMetadata': {'RequestId': 'e4452b32-18f2-4764-b388-9cd7cb9d925c',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Thu, 16 Apr 2026 22:40:50 GMT',
   'content-type': 'application/json',
   'content-length': '886',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'e4452b32-18f2-4764-b388-9cd7cb9d925c'},
  'RetryAttempts': 0},
 'citations': [{'generatedResponsePart': {'textResponsePart': {'span': {'end': 340,
      'start': 0},
     'text': 'I could not find an exact answer to the question. The search results do not contain information about what happens on "day 2" of any specific event, story, or sequence. The search results include passages about various books, video game patch notes, and other content, but none of them reference or describe events occurring on a second day.'}},
   'retrievedReferences': []}],
 'output': {'text': 'I could not find an exact answer to the question. The search results do not contain information about what happens on "day 2" of any specifi

In [15]:
df_rag_res_ans = rag_analyzer.df_rag_res.copy(deep=True)
df_rag_res_ans["source_document_index"] = df_rag_res_ans["query"].map(sp_q_doc_index_map)
df_rag_res_ans["answer"] = df_rag_res_ans["query"].map(sp_q_answer_map)


dra_cols = ['query', 'rag_answer', 'answer', 'rag_citations', 'source_document_index']
df_rag_res_ans = df_rag_res_ans[dra_cols]

df_rag_res_ans


,query,rag_answer,answer,rag_citations,source_document_index
0,What do keybullet kin drop?,Keybullet Kin drop a key upon death. Jammed Ke...,Keybullet kin drop a key upon death.,[s3://rag-search-tests/documents/doc_0.txt],doc_0.txt
1,What kind of gun does the bandana bullet kin use?,The Bandana Bullet Kin uses a Machine Pistol.,The bandana bullet kin wields a machine pistol.,[s3://rag-search-tests/documents/doc_0.txt],doc_0.txt
2,What do the giants look like?,There are two giants described. The first gian...,"One giant is burly, grey-skinned, and 20 feet ...","[s3://rag-search-tests/documents/doc_1.txt, s3...",doc_1.txt
3,What happens on day 2?,"Sorry, I am unable to assist you with this req...","After a few miles of winding tunnel, you emerg...",[],doc_1.txt
4,What were the requirements for the project?,The required features for the project were:\n-...,The tool had the following requirements:\n- Ch...,[s3://rag-search-tests/documents/doc_2.txt],doc_2.txt
5,What data did was used to test the prototype?,The prototype was not tested with any specific...,Grace Hopper's Wikipedia page and Alan Turing'...,"[s3://rag-search-tests/documents/doc_2.txt, s3...",doc_2.txt
6,How do the data storage options compare?,The data storage options are organized into di...,For fast start: use SQLite3 and ChromaDB (File...,[s3://rag-search-tests/documents/doc_3.txt],doc_3.txt
7,When was UTF-8 support added for European lang...,UTF-8 encoding for European languages was adde...,UTF-8 support was added for European languages...,[s3://rag-search-tests/documents/doc_3.txt],doc_3.txt
8,How do I make a button?,"To make a button in marimo, use `mo.ui.button(...",import marimo as mo\n\nbutton = mo.ui.run_butt...,"[s3://rag-search-tests/documents/doc_4.txt, s3...",doc_4.txt
9,When might I use caching?,You might use caching when you have expensive ...,"You might use caching when, for example, your ...",[s3://rag-search-tests/documents/doc_4.txt],doc_4.txt


In [14]:
df_rag_res_ans.columns

dra_cols = ['query', 'rag_answer', 'answer', 'rag_citations', 'source_document_index']


Index(['query', 'rag_answer', 'rag_citations', 'source_document_index',
       'answer'],
      dtype='object')